# 04 — Gene Lookup

HGNC and Ensembl BioMart reference files and columns before joining.

In [1]:
import sys, os

_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(_notebook_dir, '..', 'scripts'))

import pandas as pd
from data_utils import REF

print("REF path:", REF)

ImportError: cannot import name 'REF' from 'data_utils' (/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/src/notebooks/../scripts/data_utils.py)

In [ ]:
# Load HGNC
hgnc = pd.read_csv(os.path.join(REF, 'hgnc_complete.txt'), sep='\t', low_memory=False)
print('HGNC shape:', hgnc.shape)
print('HGNC columns:', hgnc.columns.tolist())
print()
print(hgnc.head(3))

HGNC shape: (44997, 6)
HGNC columns: ['HGNC ID', 'Approved symbol', 'Previous symbols', 'Alias symbols', 'Ensembl gene ID', 'Locus group']

      HGNC ID Approved symbol             Previous symbols  \
0      HGNC:5            A1BG                          NaN   
1  HGNC:37133        A1BG-AS1  NCRNA00181, A1BGAS, A1BG-AS   
2  HGNC:24086            A1CF                          NaN   

                       Alias symbols  Ensembl gene ID          Locus group  
0                                NaN  ENSG00000121410  protein-coding gene  
1                           FLJ23569  ENSG00000268895       non-coding RNA  
2  ACF, ASP, ACF64, ACF65, APOBEC1CF  ENSG00000148584  protein-coding gene  


In [ ]:
# Load Ensembl BioMart
ensembl = pd.read_csv(os.path.join(REF, 'ensembl_biomart.txt'), sep='\t', low_memory=False)
print('Ensembl shape:', ensembl.shape)
print('Ensembl columns:', ensembl.columns.tolist())
print()
print(ensembl.head(3))

Ensembl shape: (86411, 2)
Ensembl columns: ['Gene stable ID', 'Gene type']

    Gene stable ID Gene type
0  ENSG00000210049   Mt_tRNA
1  ENSG00000211459   Mt_rRNA
2  ENSG00000210077   Mt_tRNA


In [ ]:
# Step 1: Inner join HGNC to Ensembl on ENSG ID
gene_lookup = hgnc.merge(
    ensembl,
    left_on="Ensembl gene ID",
    right_on="Gene stable ID",
    how="inner"
)

# Step 2: Filter to protein_coding only
gene_lookup = gene_lookup[gene_lookup["Gene type"] == "protein_coding"].copy()

# Step 3: Rename columns to contract spec
gene_lookup = gene_lookup.rename(columns={
    "Ensembl gene ID":  "ensg_id",
    "Approved symbol":  "hgnc_symbol",
    "Previous symbols": "prev_symbols",
    "Alias symbols":    "alias_symbols",
    "HGNC ID":          "hgnc_id",
    "Gene type":        "biotype",
})

# Step 4: Drop columns no longer needed
gene_lookup = gene_lookup.drop(columns=["Gene stable ID", "Locus group"])

# Step 5: Convert comma-delimited lists → pipe-delimited (contract rule)
for col in ["prev_symbols", "alias_symbols"]:
    gene_lookup[col] = (
        gene_lookup[col]
        .fillna("")
        .str.strip()
        .str.replace(r",\s*", "|", regex=True)
        .replace("", float("nan"))
    )

# Step 6: Strip any ENSG version suffixes (precaution)
gene_lookup["ensg_id"] = gene_lookup["ensg_id"].str.replace(
    r"\.\d+$", "", regex=True
)

# Step 7: Final column order per contract
gene_lookup = gene_lookup[[
    "ensg_id", "hgnc_symbol", "prev_symbols",
    "alias_symbols", "hgnc_id", "biotype"
]]

# Force plain object dtype — prevents Arrow-backed StringDtype from showing as 'str'
gene_lookup = gene_lookup.astype(object)

print(gene_lookup.shape)
print(gene_lookup.dtypes)
print(gene_lookup.head(5))

(19446, 6)
ensg_id          object
hgnc_symbol      object
prev_symbols     object
alias_symbols    object
hgnc_id          object
biotype          object
dtype: object
           ensg_id hgnc_symbol prev_symbols                  alias_symbols  \
0  ENSG00000121410        A1BG          NaN                            NaN   
2  ENSG00000148584        A1CF          NaN  ACF|ASP|ACF64|ACF65|APOBEC1CF   
3  ENSG00000175899         A2M          NaN           FWP007|S863-7|CPAMD5   
5  ENSG00000166535       A2ML1       CPAMD9                  FLJ25179|p170   
8  ENSG00000184389     A3GALT2     A3GALT2P                   IGBS3S|IGB3S   

      hgnc_id         biotype  
0      HGNC:5  protein_coding  
2  HGNC:24086  protein_coding  
3      HGNC:7  protein_coding  
5  HGNC:23336  protein_coding  
8  HGNC:30005  protein_coding  


In [ ]:
print(gene_lookup.dtypes)

ensg_id          object
hgnc_symbol      object
prev_symbols     object
alias_symbols    object
hgnc_id          object
biotype          object
dtype: object


In [ ]:
print("=== VALIDATION ===")
print(f"Total rows:           {len(gene_lookup):,}")
print(f"Null ensg_id:         {gene_lookup['ensg_id'].isna().sum()}")
print(f"Duplicate ensg_id:    {gene_lookup['ensg_id'].duplicated().sum()}")
print(f"Null hgnc_symbol:     {gene_lookup['hgnc_symbol'].isna().sum()}")
print(f"Unique biotypes:      {gene_lookup['biotype'].unique()}")
print(f"Has prev_symbols:     {gene_lookup['prev_symbols'].notna().sum():,}")
print(f"Has alias_symbols:    {gene_lookup['alias_symbols'].notna().sum():,}")

=== VALIDATION ===
Total rows:           19,446
Null ensg_id:         0
Duplicate ensg_id:    0
Null hgnc_symbol:     0
Unique biotypes:      ['protein_coding']
Has prev_symbols:     7,382
Has alias_symbols:    15,704


In [ ]:
OUT = os.path.join(REF, "gene_lookup.parquet")

gene_lookup.to_parquet(OUT, index=False, engine="fastparquet")

# Confirm it saved correctly by reading it back
confirm = pd.read_parquet(OUT, engine="fastparquet")
print(f"Saved and verified: {confirm.shape}")
print(f"File size: {os.path.getsize(OUT):,} bytes")

Saved and verified: (19446, 6)
File size: 641,932 bytes


In [ ]:
# Check for duplicates and non-string column names
print("Duplicate columns:", gene_lookup.columns[gene_lookup.columns.duplicated()].tolist())
print("Column dtypes (name types):", set(type(c) for c in gene_lookup.columns))
print("Any NaN column names:", gene_lookup.columns.isna().any())


Duplicate columns: []
Column dtypes (name types): {<class 'str'>}
Any NaN column names: False
